# Concept Bottleneck Model

Instead of predicting a label straight from pixels, a **Concept Bottleneck Model (CBM)** factors the prediction through a handful of human-legible concepts:

$$x \;\xrightarrow{\,f\,}\; c \;\xrightarrow{\,g\,}\; y$$

- $f$: image &rarr; concept activations (CLIP embedding &rarr; one SVM per concept)
- $g$: concepts &rarr; class label (logistic regression)

The training is done independently (first $f$, then $g$).

Because $c$ is interpretable ("striped breast", "buff nape", ...), a human can check *why* the model predicted a class instead of trusting a black box.

We identified a hard task among the possible CUB pairs: telling apart two species that look almost identical, **Le Conte Sparrow** vs. **Savannah Sparrow**. Everything below loads straight from [`NWeak/cub-mirror`](https://huggingface.co/datasets/NWeak/cub-mirror) — images, CLIP embeddings, official CUB-200-2011 labels/concepts — no local setup needed beyond this cell.

In [1]:
from datasets import load_dataset
import torch

REPO_ID = "NWeak/cub-mirror"
ds = load_dataset(REPO_ID)

NON_CONCEPT_COLS = {
    "sample_idx", "split", "label", "class_name", "image_path", "image", "embedding", "concepts",
}
class_names = ds["train"].features["label"].names
concept_names = [c for c in ds["train"].column_names if c not in NON_CONCEPT_COLS]


def split_tensors(split):
    embeddings = torch.tensor(ds[split]["embedding"])
    embeddings = (embeddings - embeddings.mean(0, keepdim=True)) / embeddings.std(0, keepdim=True)
    concepts = torch.tensor(ds[split]["concepts"])
    labels = torch.tensor(ds[split]["label"])
    return embeddings, concepts, labels


train_embeddings, train_concepts, train_y = split_tensors("train")
val_embeddings, val_concepts, val_y = split_tensors("val")
test_embeddings, test_concepts, test_y = split_tensors("test")

print(len(class_names), "classes,", len(concept_names), "concepts")
print("train:", train_embeddings.shape, "val:", val_embeddings.shape, "test:", test_embeddings.shape)

/home/nicola.debole/projects/user-study-CBMs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


200 classes, 112 concepts
train: torch.Size([4796, 768]) val: torch.Size([1198, 768]) test: torch.Size([5794, 768])


## Step 1 — Learn the concepts ($f$)

For each of the 6 concepts picked as discriminative for this pair (striped breast, buff breast, white throat, buff nape, solid belly, brown crown), we fit an independent RBF-kernel SVM on CLIP embeddings:

$$f_j(x) = \mathrm{SVM}_j\big(\mathrm{CLIP}(x)\big) \in \mathbb{R}, \qquad j = 1, \dots, 6$$

These SVMs train on **every other species**, never on Le Conte or Savannah Sparrow, as the only two classes alone contain too few images and the risk of learning spurious features is high.

In [2]:
import numpy as np
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report

concept_mask = [23, 44, 48, 69, 89, 103]  # striped breast, buff breast, white throat, buff nape, solid belly, brown crown
SPARROW_PAIR = [123, 126]  # Le Conte Sparrow, Savannah Sparrow
masked_concept_names = [concept_names[i] for i in concept_mask]
print(masked_concept_names)

concept_model = MultiOutputClassifier(SVC(kernel="rbf", C=1.0, class_weight="balanced"))

train_other_subset = np.where(~np.isin(train_y.numpy(), SPARROW_PAIR))[0]
test_pair_subset = np.where(np.isin(test_y.numpy(), SPARROW_PAIR))[0]

concept_model.fit(
    train_embeddings[train_other_subset].numpy(),
    train_concepts[train_other_subset][:, concept_mask].numpy(),
)

concept_preds = concept_model.predict(test_embeddings[test_pair_subset].numpy())
print(classification_report(
    test_concepts[test_pair_subset][:, concept_mask].numpy(),
    concept_preds,
    target_names=masked_concept_names,
))

['striped breast', 'buff breast', 'white throat', 'buff nape', 'solid belly', 'brown crown']


                precision    recall  f1-score   support

striped breast       0.54      0.97      0.69        30
   buff breast       0.67      0.97      0.79        29
  white throat       0.82      0.77      0.79        30
     buff nape       0.41      0.59      0.49        29
   solid belly       0.93      0.45      0.60        29
   brown crown       0.50      0.30      0.38        30

     micro avg       0.60      0.67      0.64       177
     macro avg       0.64      0.67      0.62       177
  weighted avg       0.64      0.67      0.62       177
   samples avg       0.63      0.67      0.63       177



## Step 2 — Learn the label from concepts ($g$)

A single logistic regression maps the 6 ground-truth concepts — rescaled to $\{-1, +1\}$, so "absent" and "present" sit symmetrically around 0 — to the species label:

$$g(c) = \sigma\!\left(\beta_0 + \sum_{j=1}^{6} \beta_j\, c_j\right), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}$$

trained *only* on the two confusable species. Since it uses the true concepts (not $f$'s predictions), this is an upper bound and the model learns to separate them given perfect concept detection.

In [3]:
from sklearn.linear_model import LogisticRegression

train_pair_subset = np.where(np.isin(train_y.numpy(), SPARROW_PAIR))[0]
train_pair_concepts = train_concepts[train_pair_subset][:, concept_mask].numpy()
train_pair_y = train_y[train_pair_subset].numpy()

task_model = LogisticRegression(max_iter=1000, class_weight="balanced", fit_intercept=False)
task_model.fit(2 * train_pair_concepts - 1, train_pair_y)

test_pair_concepts = test_concepts[test_pair_subset][:, concept_mask].numpy()
test_pair_y = test_y[test_pair_subset].numpy()

label_preds = task_model.predict(2 * test_pair_concepts - 1)
print(classification_report(test_pair_y, label_preds))

              precision    recall  f1-score   support

         123       1.00      1.00      1.00        29
         126       1.00      1.00      1.00        30

    accuracy                           1.00        59
   macro avg       1.00      1.00      1.00        59
weighted avg       1.00      1.00      1.00        59



## Step 3 — Chain them end-to-end

$f$ and $g$ were trained independently — $f$ never saw the sparrow pair but only other bird species, while $g$ never saw an image but only concept logits. To actually *use* the CBM, we chain them:

$$\hat{y} = g\big(f(x)\big)$$

One wrinkle: $f_j(x)$ is an unbounded SVM decision value, but $g$ was trained on $\{-1, +1\}$-scaled inputs. We squash it with $\tanh$ to get a continuous, bounded concept *activation* concentrated on the extremes (-1, 1) to maintain the same scale:

$$c_j \approx \tanh\big(f_j(x)\big) \in (-1, 1)$$

In [4]:
logits = [estimator.decision_function(test_embeddings[test_pair_subset].numpy()) for estimator in concept_model.estimators_]
concept_activations = np.tanh(np.column_stack(logits))

y_pred = task_model.predict(concept_activations)
print("=== END-TO-END (image -> predicted concepts -> predicted label) ===")
print(classification_report(test_pair_y, y_pred))

weights = task_model.coef_[0]
intercept = task_model.intercept_[0]
print("weights:", weights)
print("intercept:", intercept)

=== END-TO-END (image -> predicted concepts -> predicted label) ===
              precision    recall  f1-score   support

         123       0.78      0.86      0.82        29
         126       0.85      0.77      0.81        30

    accuracy                           0.81        59
   macro avg       0.82      0.81      0.81        59
weighted avg       0.82      0.81      0.81        59

weights: [ 0.70998989 -0.70998989  0.70998989 -0.70998989 -0.70998989  0.70998989]
intercept: 0.0


## Step 4 — Package the results

For the user study we need one readable table: per test sample, the ground-truth concepts, the predicted concept activations, the label prediction, and per-concept correctness. Saved to `data/user_study/cub_user_study.csv`.

In [5]:
import pandas as pd
from pathlib import Path

test_pair_species = [class_names[label] for label in test_pair_y]
concept_activations_binary = (concept_activations > 0).astype(int)
concept_gt = test_concepts[test_pair_subset][:, concept_mask].numpy()


def build_df(gt, activations, activations_binary, labels, y_preds, species, concept_names, original_idx, split):
    base = pd.DataFrame({
        "split": split,
        "sample_idx": original_idx,
        "label": labels,
        "species": species,
        "task_pred": y_preds,
    })
    gt_df = pd.DataFrame(gt, columns=[f"{c}_gt" for c in concept_names])
    pred_df = pd.DataFrame(activations, columns=[f"{c}_pred" for c in concept_names])
    correct_df = pd.DataFrame((gt == activations_binary).astype(int), columns=[f"{c}_correct" for c in concept_names])
    return pd.concat([base, gt_df, pred_df, correct_df], axis=1)


test_df = build_df(
    concept_gt, concept_activations, concept_activations_binary, test_pair_y, y_pred,
    test_pair_species, masked_concept_names, test_pair_subset, split="test",
)

output_path = Path("../data/user_study/cub_user_study.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
test_df.to_csv(output_path, index=False)
print(f"Saved {len(test_df)} rows to {output_path}")
test_df.head()

Saved 59 rows to ../data/user_study/cub_user_study.csv


,split,sample_idx,label,species,task_pred,striped breast_gt,buff breast_gt,white throat_gt,buff nape_gt,solid belly_gt,...,white throat_pred,buff nape_pred,solid belly_pred,brown crown_pred,striped breast_correct,buff breast_correct,white throat_correct,buff nape_correct,solid belly_correct,brown crown_correct
0,test,3520,123,Le Conte Sparrow,123,0,1,0,1,1,...,-0.866693,-0.387743,-0.599325,-0.534472,0,1,1,0,0,1
1,test,3521,123,Le Conte Sparrow,123,0,1,0,1,1,...,-0.843594,0.687462,-0.341700,-0.418174,0,1,1,1,0,1
2,test,3522,123,Le Conte Sparrow,123,0,1,0,1,1,...,-0.674573,0.769137,-0.528233,0.346006,0,1,1,1,0,0
3,test,3523,123,Le Conte Sparrow,123,0,1,0,1,1,...,-0.901123,0.624179,-0.165908,-0.736422,0,1,1,1,0,1
4,test,3524,123,Le Conte Sparrow,123,0,1,0,1,1,...,-0.762420,0.038375,0.241697,-0.313701,0,1,1,1,1,1


## Step 5 — Sanity check 1

We recompute this formula by hand from the saved CSV and compare it to the model's own predictions — they should match exactly.

In [6]:
dataframe = pd.read_csv(output_path)

pred_cols = [f"{c}_pred" for c in masked_concept_names]
p_concepts = dataframe[pred_cols]


def logistic(x):
    return 1 / (1 + np.exp(-x))


def compute_prediction(activations, betas, bias):
    eta = bias
    for i, b in enumerate(betas):
        eta += b * activations.iloc[:, i]
    return logistic(eta)


toy_model = compute_prediction(p_concepts, weights, intercept)
toy_predictions = np.where(toy_model > 0.5, task_model.classes_[1], task_model.classes_[0])

diff = toy_predictions != dataframe["task_pred"].to_numpy()
print(f"Differences between the hand-computed formula and the fitted model: {diff.sum()} / {len(diff)}")

Differences between the hand-computed formula and the fitted model: 0 / 59


In [7]:
import pandas as pd
import numpy as np
from datasets import load_dataset

# Load the generated activations from the user study CSV file
generated_act = pd.read_csv('../data/user_study/cub_user_study.csv')


def col_name(stim, feature):
    return f'Stim{stim}_Feature{feature}_Detected'

def id_col_name(stim):
    return f'Stim{stim}_StimID'

def extract_id(x):
    if pd.isna(x):
        return -1
    return int(x.split('_')[1])


conc_act = generated_act[['striped breast_pred', 'buff breast_pred', 'white throat_pred',
       'buff nape_pred', 'solid belly_pred', 'brown crown_pred']]
conc_act_by_id = conc_act.set_index(generated_act['sample_idx'])


non_zero_differences = [] 
for p_row in range(len(dataframe)):
    participant_row = p_row  # row in `dataframe` to sanity-check
    stats = []
    for stimolo in range(1, 11):
        detected = dataframe[[col_name(stimolo, 1), col_name(stimolo, 2), col_name(stimolo, 3), col_name(stimolo, 4), col_name(stimolo, 5), col_name(stimolo, 6)]]
        activations_user_study = [detected.iloc[participant_row].tolist()[i] for i in [5,1,4,0,2,3]]
        #print(activations_user_study)
        activations_user_study_bool = (np.array(activations_user_study) > 0).astype(int)
        stim_id = extract_id(dataframe.loc[participant_row, id_col_name(stimolo)])
        if stim_id == -1:
            continue
        #print(stim_id)
        activations_now = conc_act_by_id.loc[stim_id].tolist()
        activations_now_bool = (np.array(activations_now) > 0).astype(int)
        #print(activations_now)
        diff = [activations_user_study[i] - activations_now[i] for i in range(len(activations_user_study))]
        diff_bool = [activations_user_study_bool[i] - activations_now_bool[i] for i in range(len(activations_user_study_bool))]
        non_zero_differences.append(np.count_nonzero(diff_bool))
        stats.append(diff)
    #stats = np.abs(np.array(stats).ravel())
    #print("Maximum numerical error", np.max(stats))

print("Total differences", np.sum(non_zero_differences))

KeyError: "None of [Index(['Stim1_Feature1_Detected', 'Stim1_Feature2_Detected',\n       'Stim1_Feature3_Detected', 'Stim1_Feature4_Detected',\n       'Stim1_Feature5_Detected', 'Stim1_Feature6_Detected'],\n      dtype='str')] are in the [columns]"